## This is the code to train the model and acquire influence for Number of Samples Experiment

**Default Code**:   
The current default code is a runnable sample. It runs on the synthetic dataset generated with sklearn's make_classification function. The default version code provides the synthetic dataset with 16500 samples and 160 features in total. The separation is set to 5 to make sure the dataset is distinguishable by the model. All features are set to be informative to ensure they are of equal importance. The dataset has only two labels, so it is a binary classification problem. The default setting will then generate the training set and test set from the pool. The default training sample size is 8000, and the test size is 500. The number of features is set to 10. The model in default will be a Simple FeedForward Neural Network constructed by TensorFlow. The Influence Estimation methods we provide by default are the Influence Function and TracIn. If you simply press 'play', the default code will generate ranked influence lists for both Influence Function and TracIn with respect to the above mentioned setting in the root directory. The result lists could then be fed into other analyses.

**By default, this is exactly the same code as the base code. Please refer to the base code for more detailed explanation.**

**Guideline**:  
Read in / Construct Datasets -> **Choose the Training Sample Size** -> Pre-processing -> Model Training -> Influence Estimation -> Store the Ranked Influence lists -> **Change the Training Sample Size and Repeat all the process** -> ... -> **After all the training and estimation, feed the results into the analysis code**  (Remember to change the file name in the last block to save lists in different settings.)

# Import Area

Here is the area to place all the import codes. You don't need to change here unless you want to customise in later sections.

In [205]:
import tensorflow as tf
import keras
from keras.utils import to_categorical
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

In [206]:
from keras import Sequential
from keras.layers import Dense, BatchNormalization, Dropout
from keras.losses import CategoricalCrossentropy
from keras.optimizers import Adam

In [207]:
from deel.influenciae.common import InfluenceModel, ExactIHVP
from deel.influenciae.influence import FirstOrderInfluenceCalculator
from deel.influenciae.utils import ORDER
from deel.influenciae.trac_in import TracIn

In [208]:
import random
from keras.optimizers import SGD

In [209]:
from sklearn.datasets import make_classification

# Dataset Construction Area

**You can use any dataset you wish here, either regression or classification. But in default, since we are using influenciae's IF and TC method, make sure they are split into train and test sets, and then stored as tensorflow dataset format. If you only want to change the dataset, you can only change the code in the first two blocks to read in/ generate your own dataset. But remember to have features X and target y before going to the third block. Also, if you wish to everything on your own, remember to add id inside the dataset.** Since our default code is for classification, the regression might need a lot of changes in all the following sections.


**Input**: Dataset chosen(Usually in Features X and Target y format)  
**Output**: Tensorflow format Train and Test Set  
**Guideline**: Input -> Turn into Dataframe and add ID -> Pre-Processing -> Change the format to Tensorflow -> Output

The default code now produces a synthetic dataset with 16500 pool, 160 features with binary classification problems. The later options will turn that into a 10 features, 8000 train set and 500 test set sample. Both sets will then be turned into TensorFlow format and will wait for training.

**The most important thing in this code is changing the train size. In this experiment, all the other things are fixed, but the number of training sample is changing to test on different number of samples.**

In [210]:
train_df_full = pd.read_csv("NoisyLabel_TrainingData.csv")
test_df = pd.read_csv("Clean_TestData.csv")

groundtruth = pd.read_csv("Noise_GroundTruth_and_TrainingLoss.csv")

foif_df = (
    pd.read_csv("NoisyLabel_FOIF_Scores.csv")
    .rename(columns={"Score": "FOIF_Score"})
)

tracin_df = (
    pd.read_csv("NoisyLabel_TracIn_Scores.csv")
    .rename(columns={"Score": "TracIn_Score"})
)

ranking_df = (
    groundtruth
    .merge(
        foif_df[["Train_ID", "FOIF_Score"]],
        on="Train_ID"
    )
    .merge(
        tracin_df[["Train_ID", "TracIn_Score"]],
        on="Train_ID"
    )
)

print(train_df_full.shape)
print(ranking_df.shape)

(8000, 15)
(8000, 7)


In [211]:
print(train_df_full.head)
print(ranking_df.head)

<bound method NDFrame.head of       feature_1  feature_2  feature_3  feature_4  feature_5  feature_6  \
0      2.000551   6.107697   0.922888  -0.645629  -2.044902   6.067214   
1     -1.482890   3.501781   1.723413  -0.987130   4.693062   3.417544   
2     -1.461155   4.709173   1.276742   2.226696   1.542537   5.980747   
3      2.349369   2.618102   0.891645  -1.675392   0.253586  -0.248629   
4     -3.087668   2.723457   0.627227   3.464238  -0.751775   0.861041   
...         ...        ...        ...        ...        ...        ...   
7995  -0.392552  -4.158898   4.090073   1.949615  -0.431500  -0.218027   
7996   3.581924  -1.483711  -0.054275  -2.953603  -1.952614  -2.313367   
7997   2.125435  -1.880934  -2.997253  -3.589638  -2.528225   0.119474   
7998  -1.806808  -0.704839   1.698431  -3.602001   0.844647   1.772143   
7999  -1.416175  -0.981882  -0.853378  -0.154625  -5.670463  -1.923880   

      feature_7  feature_8  feature_9  feature_10  label     id  clean_label  \
0

In [212]:
random_baseline_seed = 42

rng = np.random.default_rng(random_baseline_seed)

ranking_df["Random_Score"] = rng.random(len(ranking_df))
print(ranking_df.head)

<bound method NDFrame.head of       Train_ID  Clean_Label  Noisy_Label  is_noisy  Training_Loss  FOIF_Score  \
0         5018            1            1         0       0.324082    0.191415   
1        10999            1            0         1       1.027546   -0.157258   
2         3146            1            0         1       1.837602   -1.040044   
3         9147            0            0         0       0.262875    0.102674   
4        12955            1            1         0       0.520085    0.186383   
...        ...          ...          ...       ...            ...         ...   
7995     13382            0            0         0       0.135412    0.172427   
7996     13339            0            0         0       0.126330    0.125705   
7997     15658            0            0         0       0.392910   -0.200291   
7998     12979            0            0         0       0.273940    0.140438   
7999      9910            1            1         0       0.882032    0.131435  

In [213]:
# Change Here
method = "TracIn"
remove_fraction = 0.30
random_seed = 42

In [214]:
def select_ids_to_remove(
    ranking_df,
    method,
    remove_fraction,
    random_seed=42
):
    n_remove = int(len(ranking_df) * remove_fraction)

    if method == "Random":
        return (
            ranking_df
            .nsmallest(n_remove, "Random_Score")
            ["Train_ID"]
            .to_numpy()
        )

    if method == "Training Loss":
        return (
            ranking_df
            .nlargest(n_remove, "Training_Loss")["Train_ID"]
            .to_numpy()
        )

    if method == "FOIF":
        return (
            ranking_df
            .nsmallest(n_remove, "FOIF_Score")["Train_ID"]
            .to_numpy()
        )

    if method == "TracIn":
        return (
            ranking_df
            .nsmallest(n_remove, "TracIn_Score")["Train_ID"]
            .to_numpy()
        )

    if method == "No Pruning":
        return np.array([], dtype=ranking_df["Train_ID"].dtype)

    raise ValueError(f"Unknown method: {method}")

In [215]:
remove_ids = select_ids_to_remove(
    ranking_df=ranking_df,
    method=method,
    remove_fraction=remove_fraction,
    random_seed=random_seed
)

train_df = (
    train_df_full[
        ~train_df_full["id"].isin(remove_ids)
    ]
    .copy()
    .reset_index(drop=True)
)

print("Method:", method)
print("Removal rate:", remove_fraction)
print("Removed samples:", len(remove_ids))
print("Remaining samples:", len(train_df))

Method: TracIn
Removal rate: 0.3
Removed samples: 2400
Remaining samples: 5600


In [216]:
removed_info = ranking_df[
    ranking_df["Train_ID"].isin(remove_ids)
]

print(
    "Noisy samples among removed:",
    int(removed_info["is_noisy"].sum())
)

print(
    "Noisy fraction among removed:",
    removed_info["is_noisy"].mean()
)

Noisy samples among removed: 1111
Noisy fraction among removed: 0.46291666666666664


In [217]:
selected_features = sorted(
    [
        col for col in train_df.columns
        if col.startswith("feature_")
    ],
    key=lambda col: int(col.split("_")[1])
)

train_ids_original = train_df["id"].to_numpy()

IDs = (
    train_ids_original
    .reshape(-1, 1)
    .astype(np.float32)
    / 1e10
)

X_train_features = train_df[
    selected_features
].to_numpy(dtype=np.float32)

X_train = np.hstack([
    X_train_features,
    IDs
])

y_train_1d = train_df[
    "noisy_label"
].to_numpy(dtype=np.int64)

y_train = to_categorical(
    y_train_1d,
    num_classes=2
)

print(X_train.shape)
print(y_train.shape)

(5600, 11)
(5600, 2)


In [218]:
X_test = test_df.drop(columns=["label"])
y_test = test_df["label"]
IDs = X_test["id"].values.reshape(-1, 1).astype(np.float32)
IDs = IDs  / 1e10

X_test = X_test.drop(columns=["id"]).values.astype(np.float32)
X_test = np.hstack((X_test, IDs))
y_test = to_categorical(y_test.values,num_classes=2)

print(X_test.shape)
print(y_test.shape)

(500, 11)
(500, 2)


6. Now we have the train_ds and test_ds for training

In [219]:
train_ds = tf.data.Dataset.from_tensor_slices((X_train, y_train))
test_ds = tf.data.Dataset.from_tensor_slices((X_test, y_test))

# Training Area

**Could modify the model as you wish here. Again, in default, influenciae relies on TensorFlow, so use the TensorFlow model if you only want to change the model. Remember: Store the InfluenceModel into the model_list with the loss function. The InfluenceModel will be used to obtain influence later. If you don't change the estimation methods, then the final output at this step shall always be the model_list**

**Input**:Train and Test Set from Data Construction Section   
**Output**: Model List  
**Guideline**: Input -> Define the Model and Hyperparameters -> Train the Model -> Output

Always remember to train the model, get the influence model and store that in model list, unless you wish to change the estimation methods.

The default code now use the train and test set generated from the last section to train the model. The default hyperparameters are: 300 Epochs, Simple FeedForward Neural Network, CategoricalCrossEntropy Loss function, SGD optimizer. Within each epoch, the current model will be turned into an Influence Model and stored inside a model list. After the training, the model list will be passed to next section for influence estimation.

In [220]:
from tensorflow.keras.regularizers import l2

1. **Could modify the model as you wish here as long as it is tensorflow.** Just remember: Store the InfluenceModel into the model_list with the loss function

In [221]:
seed_value = 42
random.seed(seed_value)
np.random.seed(seed_value)
tf.random.set_seed(seed_value)

model = Sequential([
    Dense(16, activation='relu', input_shape=(X_train.shape[1],)),  
    BatchNormalization(momentum=0.9),
    Dropout(0.0),
    Dense(8, activation='relu'),
    Dense(y_train.shape[1])
])
loss_fn = CategoricalCrossentropy(from_logits=True)
optimizer = SGD(learning_rate=0.001, momentum=0.9)
model.compile(loss=loss_fn, optimizer=optimizer, metrics=['accuracy'])

epochs = 300
unreduced_loss_fn = CategoricalCrossentropy(from_logits=True, reduction=tf.keras.losses.Reduction.NONE)
model_list = []
model_list.append(InfluenceModel(model, start_layer=-1, loss_function=unreduced_loss_fn))
for i in range(epochs):
  model.fit(train_ds.batch(256), epochs=1, validation_data=test_ds.batch(256), verbose=2)
  model_list.append(InfluenceModel(model, start_layer=-1, loss_function=unreduced_loss_fn))
base_loss, acc = model.evaluate(test_ds.batch(32), verbose=2)
print(base_loss)

22/22 - 1s - loss: 0.9460 - accuracy: 0.5389 - val_loss: 0.8895 - val_accuracy: 0.4620 - 550ms/epoch - 25ms/step
22/22 - 0s - loss: 0.7560 - accuracy: 0.5505 - val_loss: 0.6973 - val_accuracy: 0.5600 - 54ms/epoch - 2ms/step
22/22 - 0s - loss: 0.6475 - accuracy: 0.6193 - val_loss: 0.6048 - val_accuracy: 0.6560 - 47ms/epoch - 2ms/step
22/22 - 0s - loss: 0.5901 - accuracy: 0.6938 - val_loss: 0.5547 - val_accuracy: 0.7300 - 53ms/epoch - 2ms/step
22/22 - 0s - loss: 0.5518 - accuracy: 0.7487 - val_loss: 0.5214 - val_accuracy: 0.7740 - 50ms/epoch - 2ms/step
22/22 - 0s - loss: 0.5216 - accuracy: 0.7802 - val_loss: 0.4952 - val_accuracy: 0.8120 - 47ms/epoch - 2ms/step
22/22 - 0s - loss: 0.4952 - accuracy: 0.8068 - val_loss: 0.4727 - val_accuracy: 0.8320 - 60ms/epoch - 3ms/step
22/22 - 0s - loss: 0.4710 - accuracy: 0.8277 - val_loss: 0.4523 - val_accuracy: 0.8540 - 76ms/epoch - 3ms/step
22/22 - 0s - loss: 0.4484 - accuracy: 0.8405 - val_loss: 0.4338 - val_accuracy: 0.8620 - 55ms/epoch - 3ms/step

# Influence Estimation Area

**Again, you could use other influence analysis methods rather than IF/TC. You can also use any other Influence Function or TracIn implementation. Just Remember: 1. Make sure the package is unform throughout the framework. 2. Generate a Ranked influence list for each Influence Function and TracIn; Only the ranked influence list could be fed into the following analysis code.**

**The default code now use the model list, train set and test set to estimate the influence, and produce a ranked influence list for both IF and TC. The results are then saved in the root directory.**

**Input**:Model list from Training section, Train and Test Set from Data Construction Section   
**Output**: Two ranked Influence Lists for IF and TC.  
**Guideline**: Input -> Influence Estimation Methods -> Influence Matrix -> Output

1. Influence Function: Here we use the influenciae package. The following code will directly generate the influence list. If you wish to have the matrix, just use influence_matrix.

2. TracIn: Here we use the influenciae package. The following code will directly generate the influence list. If you wish to have the matrix, just use TracIn_matrix.

3. Here we turn both influence lists to the ranked influence lists and then store them for further processing.